## 문서 관리 정책

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day03" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w6" / "day03"

if str(ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(ROOT / "backend"))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

from app.core.config import get_settings

url = get_settings().database_url
print("프로젝트 루트 :", ROOT)
print("붙은 DB       :", url.split("@")[-1] if "@" in url else url)

프로젝트 루트 : c:\workspace\hanwha-agent
붙은 DB       : localhost:5432/agent


In [2]:
import shutil                       

from sqlalchemy import func, select

from app.db.seed_data import DOCUMENTS      
from app.db.session import session_scope
from app.models.document import Chunk, DocumentVersion
from app.services import document_service

MATERIALS = next((p for p in (SAMPLES, ROOT.parent / "seed_docs") if p.exists()), None)
if MATERIALS is None:                    
    print(" 재료 폴더가 없습니다 — sandbox\\w5\\day01\\samples 도, 프로젝트 폴더 옆 seed_docs 도 없습니다.")
    print("   이미 uploads/ 에 있는 파일만 적재합니다. \n")

rows = []
for doc in DOCUMENTS:
    for v in doc["versions"]:
        dst = ROOT / v["file_path"]         
        if not dst.exists() and MATERIALS is not None:
            src = next(MATERIALS.glob(f"{doc['id']}_*_{v['version']}.{v['file_format']}"), None)
            if src is None:
                print(f"  {doc['id']} {v['version']} : 재료가 없습니다 — 이 줄만 건너뜁니다")
                continue
            dst.parent.mkdir(exist_ok=True)
            shutil.copy2(src, dst)           
        with session_scope() as s:
            dv = s.scalar(select(DocumentVersion).where(
                DocumentVersion.doc_id == doc["id"], DocumentVersion.version == v["version"]))
            real = s.scalar(select(func.count()).select_from(Chunk)
                            .where(Chunk.version_id == dv.id)) if dv else 0
        if dv is not None and real == 0 and dst.exists():
            document_service.ingest_document(doc_id=doc["id"], version=v["version"], path=str(dst))
        rows.append((doc["id"], v["version"], v["file_format"], v["status"], v["chunk_count"]))

total = table_chunks = 0
with session_scope() as s:
    for doc_id, version, fmt, status, claimed in sorted(rows):
        dv = s.scalar(select(DocumentVersion).where(
            DocumentVersion.doc_id == doc_id, DocumentVersion.version == version))
        real = s.scalar(select(func.count()).select_from(Chunk).where(Chunk.version_id == dv.id))
        total += real
        table_chunks += s.scalar(select(func.count()).select_from(Chunk)
                                 .where(Chunk.version_id == dv.id, Chunk.kind == "표"))
        print(f"{doc_id} {version:<5}{fmt:<6}{status:<5} 청크 {real:>3}   (시드가 적어 둔 값 {claimed:>3})")

print()
print(f"여덟 줄 · 청크 합계 {total}  (시드의 주장 합계 {sum(r[4] for r in rows)})  ·  표 청크 {table_chunks}")

c:\workspace\hanwha-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


09:32:34 INFO     sentence_transformers.base.model No device provided, using cpu
09:32:34 INFO     sentence_transformers.base.model Loading SentenceTransformer model from ..\models\bge-m3.


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 49184.92it/s]


09:32:38 INFO     app.integrations.local_embed 로컬 임베딩 모델 준비: ..\models\bge-m3 / 1024차원


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.47s/it]


...

09:35:38 INFO     app.rag.store 청크 적재 : DOC-SE-003 v2.2 - 20청크 - 20벡터



Batches: 100%|██████████| 1/1 [00:12<00:00, 12.56s/it]

09:35:51 INFO     app.rag.store 청크 적재 : DOC-SE-011 v1.0 - 21청크 - 21벡터



Batches: 100%|██████████| 1/1 [00:16<00:00, 16.41s/it]

09:36:08 INFO     app.rag.store 청크 적재 : DOC-PM-005 v2.0 - 17청크 - 17벡터
DOC-HR-002 v3.1 pdf   현행    청크  24   (시드가 적어 둔 값 112)
DOC-HR-014 v1.1 docx  만료    청크  24   (시드가 적어 둔 값  79)
DOC-HR-014 v2.0 docx  현행    청크  53   (시드가 적어 둔 값  84)
DOC-HR-021 v1.2 docx  현행    청크  21   (시드가 적어 둔 값  41)
DOC-PM-005 v2.0 pdf   현행    청크  17   (시드가 적어 둔 값  87)
DOC-PU-007 v4.0 pdf   현행    청크  25   (시드가 적어 둔 값 156)
DOC-SE-003 v2.2 pdf   현행    청크  20   (시드가 적어 둔 값  98)
DOC-SE-011 v1.0 docx  현행    청크  21   (시드가 적어 둔 값  33)

여덟 줄 · 청크 합계 205  (시드의 주장 합계 690)  ·  표 청크 15


In [3]:
from app.db.session import session_scope
from app.rag import embedder, store 

QUESTION = "출장 갈 때 숙박비 얼마 나와요?"

query_vec = embedder.embed_query(QUESTION)
with session_scope() as s:
    hits = store.vector_search(s, query_vec, limit=12) 

print(f"질문 : {QUESTION} -> {len(hits)}건")
print("#\t\t문서\t\t\t판\t\t상태\t\t등급\t\t점수\t\t조각")
for i, h in enumerate(hits, 1):
    print(f"{i}  {h['title']}  {h['version']}  {h['version_status']}  {h['security_level']}  {h['score']}  {h['locator']}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.88it/s]

질문 : 출장 갈 때 숙박비 얼마 나와요? -> 12건
#		문서			판		상태		등급		점수		조각
1  국내출장 여비 규정  v2.0  현행  일반  0.6901  제3장 제2절 제14조(숙박비) · 별표1
2  국내출장 여비 규정  v1.1  만료  일반  0.6591  제3장 제9조(식비)
3  국내출장 여비 규정  v2.0  현행  일반  0.6582  제3장 제1절 제12조(식비) · 별표1
4  국내출장 여비 규정  v2.0  현행  일반  0.6555  표2
5  국내출장 여비 규정  v2.0  현행  일반  0.6492  표3
6  국내출장 여비 규정  v2.0  현행  일반  0.6465  제4장 제22조(장기 출장)
7  국내출장 여비 규정  v2.0  현행  일반  0.6445  제2장 제6조(출장의 구분)
8  국내출장 여비 규정  v2.0  현행  일반  0.6422  제3장 제2절 제16조(숙박비의 예외)
9  국내출장 여비 규정  v2.0  현행  일반  0.6335  제3장 제1절 제11조(일비) · 별표1
10  국내출장 여비 규정  v1.1  만료  일반  0.6284  제3장 제11조(숙박비) · 별표1
11  국내출장 여비 규정  v1.1  만료  일반  0.6244  제2장 제5조(출장의 구분)
12  국내출장 여비 규정  v1.1  만료  일반  0.6199  제1장 제2조(정의)
